# 📚 Smart Document Q&A with RAG
**Retrieval-Augmented Generation for Legal/Technical Documents**

_End-to-End RAG System — Self-Hosted with Chroma & Open Source Models_

---

## What I Learned in Week 5

Week 5 covered the core building blocks of RAG (Retrieval-Augmented Generation):
- **RAG fundamentals**: Combining LLM with knowledge base for accurate Q&A
- **Chroma vector database**: Storing embeddings for semantic search
- **Text embeddings**: Converting text to numerical representations
- **Document chunking**: Splitting documents for optimal retrieval
- **Semantic search**: Finding relevant docs by meaning, not keywords
- **Evaluation**: Measuring RAG system accuracy (evals)
- **LangChain 1.0**: Modern RAG pipeline construction

## Why This Project

As a developer, I need a tool that can:
1. Answer questions about my codebase/contracts/docs
2. Cite sources for every answer
3. Work with my own documents (no API training)
4. Provide accurate answers grounded in context
5. Run locally for privacy

This exercise builds a production-ready RAG system for legal/technical documents.

---

## Components Used

| Component | Purpose |
|---|---|
| `crawler` (requests + BeautifulSoup) | Same-domain crawl, 200-link limit, stats |
| `bertopic` + `transformers` (zero-shot) | BERTopic topics, then zero-shot per topic for category name |
| `chromadb` | Persistent vector database at data/db |
| `OpenAIEmbeddings` | Convert text to vectors |
| `RecursiveCharacterTextSplitter` | Smart document chunking |
| `OpenAI` client | LLM for answer generation |
| `gr.Blocks` | Interactive Q&A with source URL citations |

---

## How It Works

1. **Crawl** → Visit website, follow same-domain links (max 200), extract content; BERTopic on all pages for content-based categories (zero-shot topic names). Re-run to re-categorize existing crawl.
2. **Store** → Save to `data/source/<domain>/<category>/` with source URL per page; optional _summary.md
3. **Load & Chunk** → Read from data/source, split into semantic sections (~500 chars each)
3. **Embed** → Convert chunks to vectors using OpenAI
4. **Store** → Save to Chroma at `data/db`
5. **Query** → User asks question, convert to embedding
6. **Retrieve** → Find similar chunks using cosine similarity
7. **Generate** → Pass context + question to LLM
8. **Respond** → Answer with source URL citations

---

## RAG Pipeline Architecture

```
[User Query]
       ↓
[Embed Query] → OpenAI Embeddings
       ↓
[Semantic Search] → Chroma (cosine similarity)
       ↓
[Top-K Chunks] → Retrieved context
       ↓
[Generate Answer] → LLM with context
       ↓
[Response + Sources] → Final output
```

---

**Hardware:** Runs on CPU with optional GPU for embeddings

## Cell 1 — Install Dependencies

In [3]:
# Run once — restart kernel after installation
!uv pip install -q chromadb langchain-openai langchain-core tiktoken gradio plotly scikit-learn requests beautifulsoup4 transformers bertopic sentence-transformers umap-learn hdbscan

print("Dependencies installed. Restart kernel if needed.")

Dependencies installed. Restart kernel if needed.


## Cell 2 — Configuration & Imports

In [12]:
import os
import glob
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
import gradio as gr
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go

# Configuration
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key: {openai_api_key[:8]}...")
else:
    print("⚠️ No OpenAI key found!")

MODEL = "gpt-4.1-nano"
EMBEDDING_MODEL = "text-embedding-3-large"
COLLECTION_NAME = "documents"
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50

DATA_DIR = Path("data")
DATA_SOURCE = DATA_DIR / "source"
DB_PATH = DATA_DIR / "db"
BASE_URL = "https://topfaith.sch.ng"

openai_client = OpenAI()
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)


print("✅ Configuration ready.")

OpenAI API Key: sk-proj-...
✅ Configuration ready.


## Cell 3 — Knowledge Base from Website Crawl

In [10]:
from urllib.parse import urlparse
import re
import shutil
import numpy as np
from crawler import crawl_site
from transformers import pipeline
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN

def get_domain(url: str) -> str:
    return urlparse(url).netloc or "unknown"

def sanitize_category(name: str) -> str:
    s = (name or "").lower().replace(" ", "_").replace("-", "_").strip()
    s = re.sub(r"[^a-z0-9_]", "", s)[:30]
    return s or "general"

CATEGORY_STOPLIST = frozenset({
    "and", "the", "for", "are", "but", "not", "you", "all", "can", "had", "her", "his", "was", "one",
    "our", "out", "has", "how", "its", "may", "now", "own", "say", "see", "way", "who", "did", "get",
    "click", "read", "more", "here", "page", "form", "back", "next", "prev", "view", "link", "menu",
})
MIN_CATEGORY_LEN = 3

def is_number_or_contact_like(sanitized: str) -> bool:
    if not sanitized:
        return False
    return all(c.isdigit() for c in sanitized)

def is_meaningful_category(sanitized: str) -> bool:
    if len(sanitized) < MIN_CATEGORY_LEN or sanitized in CATEGORY_STOPLIST:
        return False
    if is_number_or_contact_like(sanitized):
        return False
    return True

def filter_topic_words(words_tuples: list, max_words: int = 5) -> list:
    if not words_tuples:
        return []
    allowed = [w for w, _ in words_tuples if len(w) >= MIN_CATEGORY_LEN and w.lower() not in CATEGORY_STOPLIST and not is_number_or_contact_like(re.sub(r"[^a-z0-9]", "", w.lower()))]
    return allowed[:max_words]

def load_pages_from_md(domain_dir: Path) -> list[dict]:
    pages = []
    for md_path in domain_dir.rglob("*.md"):
        if md_path.name == "_summary.md":
            continue
        raw = md_path.read_text(encoding="utf-8")
        url, title, text = "", "No title", raw
        if raw.startswith("Source: "):
            first_line, _, rest = raw.partition("\n\n")
            url = first_line.replace("Source: ", "").strip()
            parts = rest.split("\n\n", 1)
            if parts and parts[0].startswith("# "):
                title = parts[0].replace("# ", "").strip()
                text = parts[1] if len(parts) > 1 else ""
            else:
                text = rest
        pages.append({"url": url, "title": title, "text": text})
    return pages

def categorize_and_save(pages: list[dict], domain_dir: Path) -> None:
    docs_for_bertopic = [(p["title"] + " " + (p["text"] or "")[:500])[:512] for p in pages]
    hdbscan_model = HDBSCAN(min_cluster_size=4, min_samples=2, cluster_selection_method="eom", prediction_data=True)
    embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
    topic_model = BERTopic(embedding_model=embedding_model, hdbscan_model=hdbscan_model)
    topics, probs = topic_model.fit_transform(docs_for_bertopic)
    topics = np.asarray(topics)
    if probs is not None:
        probs = np.asarray(probs)
    classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli")
    topic_label_map = {-1: "general"}
    unique_topics = sorted(set(topics))
    for topic_id in unique_topics:
        if topic_id == -1:
            continue
        words_tuples = topic_model.get_topic(topic_id) or []
        words = filter_topic_words(words_tuples, max_words=5)
        if not words:
            topic_label_map[topic_id] = "general"
            continue
        indices = np.where(topics == topic_id)[0]
        if probs is not None and probs.ndim == 2 and topic_id < probs.shape[1]:
            j = int(indices[np.argmax(probs[indices, topic_id])])
        else:
            j = int(indices[0])
        rep_doc = docs_for_bertopic[j]
        out = classifier(rep_doc, words, multi_label=False)
        chosen = "general"
        for label in out["labels"]:
            cand = sanitize_category(label)
            if is_meaningful_category(cand):
                chosen = cand
                break
        if is_number_or_contact_like(chosen):
            chosen = "contacts"
        topic_label_map[topic_id] = chosen
    category_counters = {}
    for idx, page in enumerate(pages):
        tid = int(topics[idx])
        cat = topic_label_map.get(tid, "general")
        cat_dir = domain_dir / cat
        cat_dir.mkdir(parents=True, exist_ok=True)
        category_counters[cat] = category_counters.get(cat, 0) + 1
        n = category_counters[cat]
        path = cat_dir / f"page_{n}.md"
        content = f"Source: {page['url']}\n\n# {page['title']}\n\n{page['text']}"
        path.write_text(content, encoding="utf-8")
    n_cats = len(category_counters)
    print(f"Saved {len(pages)} pages under {domain_dir}: {n_cats} content-based categories")

domain = get_domain(BASE_URL)
domain_dir = DATA_SOURCE / domain
domain_dir.mkdir(parents=True, exist_ok=True)

md_files = [p for p in domain_dir.glob("**/*.md") if p.name != "_summary.md"]
if md_files:
    print(f"Re-categorizing {len(md_files)} existing pages at {domain_dir} (content-based)...")
    pages = load_pages_from_md(domain_dir)
    for subdir in domain_dir.iterdir():
        if subdir.is_dir():
            shutil.rmtree(subdir)
    if not pages:
        pages = [{"url": BASE_URL, "title": "Demo", "text": "No content to re-categorize."}]
        (domain_dir / "general").mkdir(exist_ok=True)
        (domain_dir / "general" / "page_1.md").write_text(
            f"Source: {BASE_URL}\n\n# Demo\n\n{pages[0]['text']}", encoding="utf-8"
        )
        print("No pages loaded; created general/page_1.md")
    else:
        categorize_and_save(pages, domain_dir)
else:
    print(f"Crawling {BASE_URL} (max 200 links, same-domain only)...")
    pages, stats = crawl_site(BASE_URL, output_dir=domain_dir, max_links=200)
    print(f"Pages visited: {stats['total_pages_visited']}, Links discovered: {stats['total_links_discovered']}")
    if not pages:
        fallback_dir = domain_dir / "general"
        fallback_dir.mkdir(exist_ok=True)
        (fallback_dir / "demo.md").write_text(
            "Source: https://example.com\n\n# Demo\n\nNo pages could be crawled. Using demo content for RAG.",
            encoding="utf-8",
        )
        print("No pages crawled; created demo.md for RAG.")
    else:
        categorize_and_save(pages, domain_dir)

if (domain_dir / "_summary.md").exists():
    print((domain_dir / "_summary.md").read_text(encoding="utf-8"))

Crawling https://topfaith.sch.ng (max 200 links, same-domain only)...
Pages visited: 28, Links discovered: 27


Device set to use mps:0


Saved 28 pages under data/source/topfaith.sch.ng: 4 content-based categories
# Crawl summary

- Domain: topfaith.sch.ng
- Crawl date: 2026-03-02 23:08 UTC
- Total pages visited: 28
- Total links discovered (same-domain): 27



## Cell 4 — Document Processing & Embedding

In [13]:
from langchain_community.docstore.document import Document

documents = []
for md_path in domain_dir.rglob("*.md"):
    if md_path.name == "_summary.md":
        continue
    raw = md_path.read_text(encoding="utf-8")
    url = ""
    if raw.startswith("Source: "):
        first_line, _, body = raw.partition("\n\n")
        url = first_line.replace("Source: ", "").strip()
        raw = body
    category = md_path.parent.name
    doc = Document(
        page_content=raw,
        metadata={"source": md_path.name, "url": url, "category": category},
    )
    documents.append(doc)

print(f"📄 Loaded {len(documents)} documents from {domain_dir}")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    length_function=len,
)
chunks = text_splitter.split_documents(documents)
print(f"✂️ Split into {len(chunks)} chunks")

for i, chunk in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i} ---")
    print(f"Source: {chunk.metadata.get('source')}, URL: {chunk.metadata.get('url', '')[:50]}...")
    print(f"Content: {chunk.page_content[:200]}...")

📄 Loaded 28 documents from data/source/topfaith.sch.ng
✂️ Split into 141 chunks

--- Chunk 0 ---
Source: page_1.md, URL: https://topfaith.sch.ng/pg/admissions/admissions-g...
Content: # Admissions guide | Topfaith Schools...

--- Chunk 1 ---
Source: page_1.md, URL: https://topfaith.sch.ng/pg/admissions/admissions-g...
Content: +234 706 6211 122
+234 803 454 6317
info@topfaith.sch.ng
Admissions Guide
Application Forms for Admissions Are on Sale.
Application forms for entrance examinations into JS1 and Transfers are now on sa...

--- Chunk 2 ---
Source: page_1.md, URL: https://topfaith.sch.ng/pg/admissions/admissions-g...
Content: click here
For the 2025/2026 Admissions' List,
click here
You can also visit any of the sales venues below.
Form Sales Venues:
(a) Online:
Fill the Application
Form
online and pay with your ATM Card
(...


## Cell 5 — Chroma Vector Store

In [14]:
from chromadb import PersistentClient

DB_PATH.mkdir(parents=True, exist_ok=True)
client = PersistentClient(path=str(DB_PATH))

if not chunks:
    print("⚠️ No chunks to index; loading existing vector store if present.")
    vectorstore = Chroma(client=client, collection_name=COLLECTION_NAME, embedding_function=embeddings)
else:
    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass
    print("🆕 Building vector store from chunks...")
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        client=client,
        collection_name=COLLECTION_NAME,
    )
print(f"✅ Vector store ready at {DB_PATH} with {vectorstore._collection.count()} embeddings")

🆕 Building vector store from chunks...
✅ Vector store ready at data/db with 141 embeddings


## Cell 6 — Retrieval & Answer Generation

In [17]:
def retrieve_and_answer(query: str, top_k: int = 3) -> dict:
    """
    Complete RAG pipeline: retrieve relevant chunks and generate answer.
    
    Args:
        query: User's question
        top_k: Number of chunks to retrieve
    
    Returns:
        Dictionary with answer and sources
    """
    # Step 1: Retrieve relevant documents
    docs = vectorstore.similarity_search(query, k=top_k)
    
    # Step 2: Build context from retrieved docs
    context = "\n\n".join([doc.page_content for doc in docs])
    sources = [doc.metadata.get("url") or doc.metadata.get("source", "") for doc in docs]
    
    # Step 3: Generate answer with LLM
    system_prompt = """You are a helpful assistant answering questions based on provided documentation.
    Always cite your sources. If you don't know the answer, say so.
    
    Context from documentation:
{context}
"""
    
    messages = [
        {"role": "system", "content": system_prompt.format(context=context)},
        {"role": "user", "content": query}
    ]
    
    response = openai_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.3,
        max_tokens=500,
    )
    
    answer = response.choices[0].message.content
    
    # Step 4: Format sources (prefer URL for reference)
    unique_sources = list(dict.fromkeys(sources))
    sources_text = "\n".join([f"- {s}" for s in unique_sources if s])
    
    return {
        "answer": answer,
        "sources": sources_text,
        "num_chunks_retrieved": len(docs)
    }


# Test the pipeline
test_query = "How much is the fees?"
print(f"❓ Query: {test_query}\n")

result = retrieve_and_answer(test_query)
print(f"💬 Answer:\n{result['answer']}")
print(f"\n📚 Sources:\n{result['sources']}")

❓ Query: How much is the fees?

💬 Answer:
The fees vary depending on the class and term. For example, the fees for JS1 students are ₦620,000 for the first term and ₦550,000 for the second and third terms. For SS3 students, the fee is ₦708,000 for the first term. The fees cover tuition, feeding, accommodation, uniforms, shoes, snacks, first aid, text/exercise books, cutlery, and beddings.

📚 Sources:
- https://topfaith.sch.ng/pg/admissions/school-fees
- https://topfaith.sch.ng/payment/fees
- https://topfaith.sch.ng/


## Cell 7 — Gradio UI

In [ ]:
def rag_pipeline(query: str):
    """Gradio handler for RAG pipeline."""
    if not query.strip():
        return "⚠️ Please enter a question.", ""
    
    try:
        result = retrieve_and_answer(query)
        response = f"{result['answer']}\n\n---\n\n**Sources:**\n{result['sources']}"
        return response, f"Retrieved {result['num_chunks_retrieved']} relevant chunks"
    except Exception as e:
        return f"❌ Error: {str(e)}", ""


with gr.Blocks(title="📚 Smart Document Q&A") as demo:
    gr.Markdown("""# 📚 Smart Document Q&A with RAG
    
    **Ask questions about our documentation**
    
    This system uses Retrieval-Augmented Generation to answer questions
    based on the knowledge base. Answers include source citations.
    """)
    
    with gr.Row():
        query_input = gr.Textbox(
            label="❓ Your Question",
            placeholder="e.g., What is the pricing for the Pro tier?",
            lines=2
        )
    
    ask_btn = gr.Button("🔍 Get Answer", variant="primary")
    
    with gr.Row():
        answer_output = gr.Markdown(label="💬 Answer")
    
    status_output = gr.Markdown(label="📊 Status")
    
    # Example questions
    gr.Examples(
        examples=[
            ["How do I authenticate my API requests?"],
            ["What is the pricing for the Pro tier?"],
            ["How do I create a new user?"],
            ["What happens if I exceed my rate limit?"],
            ["What payment methods do you accept?"],
        ],
        inputs=[query_input],
    )
    
    ask_btn.click(
        rag_pipeline,
        inputs=[query_input],
        outputs=[answer_output, status_output]
    )

demo.launch(inbrowser=True)

---

## Concepts Demonstrated

| Cell | Pipeline / API | What it shows |
|------|---------------|---------------|
| 3 | Crawl + HF Categorize | Same-domain crawl, stats, zero-shot category, data/source |
| 4 | Document Splitting | Load from data/source, RecursiveCharacterTextSplitter |
| 5 | Chroma DB | Vector storage in data/db |
| 6 | RAG Pipeline | Full retrieval + generation |
| 6 | Citations | Source attribution |
| 7 | Gradio UI | Interactive Q&A interface |

---

## Extensions Possible

1. **Hybrid search**: Combine semantic + keyword search
2. **Evaluation**: Add RAGAS metrics for accuracy
3. **Re-ranking**: Improve chunk selection
4. **Multi-modal**: Support PDFs, images
5. **Streaming**: Stream LLM responses
6. **Conversation history**: Follow-up questions
7. **Custom embeddings**: Use local embedding models
